## Load Bundesliga JSON-LD into Neo4j

Imports the JSON-LD file (ontology + data from `rag_llm.ipynb`) into Neo4j via n10s and validates that all relationship types are present.

**Requirements:** Neo4j running, [Neosemantics (n10s)](https://neo4j.com/labs/neosemantics/) plugin installed.

In [ ]:
# --- Step 0: Install dependencies (run once) ---
!pip install -q neo4j rdflib

In [24]:
# --- Step 1: Configuration ---
from pathlib import Path

NEO4J_URI = "neo4j://127.0.0.1:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "fcdf2026"

project_root = Path(".").resolve()
rdf_file = project_root / "Bundesliga23_24_test.jsonld"

print(f"JSON-LD: {rdf_file.name} (exists: {rdf_file.exists()})")
print(f"Neo4j: {NEO4J_URI}")

JSON-LD: Bundesliga23_24_test.jsonld (exists: True)
Neo4j: neo4j://127.0.0.1:7687


### Import and validate

Clears the graph, imports the TTL, then validates that all relationship types from the source are present in Neo4j.

In [26]:
# --- Import and validate ---
from neo4j import GraphDatabase
from rdflib import Graph, URIRef

def import_via_n10s(uri, user, password, rdf_path):
    driver = GraphDatabase.driver(uri, auth=(user, password))
    file_url = "file:///" + str(rdf_path.resolve()).replace("\\", "/")
    fmt = "JSON-LD" if rdf_path.suffix == ".jsonld" else "Turtle"
    with driver.session() as session:
        session.run("CREATE CONSTRAINT n10s_unique_uri IF NOT EXISTS FOR (r:Resource) REQUIRE r.uri IS UNIQUE")
        session.run("CALL n10s.graphconfig.init()")
        result = session.run("CALL n10s.rdf.import.fetch($url, $format)", {"url": file_url, "format": fmt})
        rec = result.single()
        print(f"n10s import: {rec.get('triplesLoaded', 0)} triples loaded")
    driver.close()

def validate_relationships(rdf_path, uri, user, password):
    g = Graph()
    fmt = "json-ld" if rdf_path.suffix == ".jsonld" else "turtle"
    g.parse(rdf_path, format=fmt)
    expected = {}
    for s, p, o in g:
        if isinstance(o, URIRef):
            pred_uri = str(p)
            if "rdf-syntax-ns#type" in pred_uri:
                continue
            local = pred_uri.split("#")[-1] if "#" in pred_uri else pred_uri.split("/")[-1]
            expected[local] = pred_uri

    driver = GraphDatabase.driver(uri, auth=(user, password))
    with driver.session() as session:
        actual = {r["relationshipType"] for r in session.run("CALL db.relationshipTypes() YIELD relationshipType")}
    driver.close()

    missing = []
    for local, full_uri in sorted(expected.items()):
        norm = local.replace("-", "_")
        match = any(r.endswith(norm) or r.endswith("__" + norm) or norm in r.split("__")[-1] for r in actual)
        if not match:
            missing.append((local, full_uri))

    print("\n=== Validation ===")
    print(f"Expected: {len(expected)} | Missing: {len(missing)}")
    if missing:
        for local, uri in missing:
            print(f"  - {local}")
        return False
    print("All relationship types present.")
    return True

# Clear, import, validate
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
driver.close()

import_via_n10s(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, rdf_file)
ok = validate_relationships(rdf_file, NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
if not ok:
    raise RuntimeError("Validation failed: missing relationship types.")
print("\nDone. Open Neo4j Browser to explore.")

n10s import: 202531 triples loaded

=== Validation ===
Expected: 25 | Missing: 0
All relationship types present.

Done. Open Neo4j Browser to explore.
